# LLM-Powered Analytics Assistant — EDA & Pipeline Evaluation
**GUVI / HCL Final Project** | Brazilian Olist E-Commerce Dataset (2016–2018)

This notebook covers:
1. Dataset overview & null analysis
2. Orders & revenue analysis
3. Review & sentiment analysis
4. Delivery performance analysis
5. Customer & seller analysis
6. Pipeline evaluation (SQL / RAG / HYBRID)

## 1. Setup & Data Loading

In [ ]:
import os, sys, sqlite3, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

warnings.filterwarnings("ignore")
sys.path.insert(0, "D:/workspace/LLM-PoweredAnalytics")

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

print("Libraries loaded ✓")

In [ ]:
DB_PATH = "D:/workspace/LLM-PoweredAnalytics/database/olist.db"
conn    = sqlite3.connect(DB_PATH)

tables = ["orders", "order_items", "order_reviews",
          "order_payments", "customers", "products",
          "sellers", "product_category"]

dfs = {t: pd.read_sql_query(f"SELECT * FROM {t}", conn) for t in tables}

for name, df in dfs.items():
    print(f"{name:30} → {df.shape[0]:>8,} rows  {df.shape[1]:>2} cols")

## 2. Dataset Overview

In [ ]:
# Null analysis per table
print("=== NULL COUNTS PER TABLE ===\n")
for name, df in dfs.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f"── {name}")
        for col, cnt in nulls.items():
            print(f"   {col:45} {cnt:>7,}  ({cnt/len(df)*100:.1f}%)")
        print()

In [ ]:
# Key business metrics
orders   = dfs["orders"]
reviews  = dfs["order_reviews"]
items    = dfs["order_items"]
payments = dfs["order_payments"]
products = dfs["products"]
sellers  = dfs["sellers"]
customers= dfs["customers"]
cat_trans= dfs["product_category"]

print("=" * 45)
print("       KEY DATASET METRICS")
print("=" * 45)
print(f"  Total orders:         {len(orders):>10,}")
print(f"  Total revenue:    R$ {items['price'].sum():>12,.2f}")
print(f"  Total reviews:        {len(reviews):>10,}")
print(f"  Reviews with text:    {reviews['review_comment_message'].notna().sum():>10,}")
print(f"  Avg review score:     {reviews['review_score'].mean():>10.2f} / 5")
print(f"  Unique customers:     {orders['customer_id'].nunique():>10,}")
print(f"  Unique sellers:       {sellers['seller_id'].nunique():>10,}")
print(f"  Unique products:      {products['product_id'].nunique():>10,}")
print(f"  Product categories:   {products['product_category_name'].nunique():>10,}")
print("=" * 45)

## 3. Orders Analysis

In [ ]:
# Order status distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

status_counts = orders["order_status"].value_counts()
status_counts.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Order Status Distribution")
axes[0].set_xlabel("Status")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

axes[1].pie(status_counts.values, labels=status_counts.index,
            autopct="%1.1f%%", startangle=90)
axes[1].set_title("Order Status Share")

plt.tight_layout()
plt.show()
print(status_counts.to_string())

In [ ]:
# Monthly order volume over time
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["month"] = orders["order_purchase_timestamp"].dt.to_period("M")

monthly = orders.groupby("month").size().reset_index(name="count")
monthly["month"] = monthly["month"].astype(str)

fig = px.line(monthly, x="month", y="count",
              title="Monthly Order Volume (2016–2018)",
              labels={"count": "Orders", "month": "Month"},
              markers=True)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 4. Revenue Analysis

In [ ]:
# Top 15 categories by revenue
prod_cat = products[["product_id","product_category_name"]].merge(cat_trans, on="product_category_name", how="left")
revenue  = (items
    .merge(prod_cat, on="product_id")
    .groupby("product_category_name_english")["price"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index())

fig = px.bar(revenue, x="product_category_name_english", y="price",
             title="Top 15 Product Categories by Revenue",
             labels={"price": "Revenue (R$)", "product_category_name_english": "Category"},
             color="price", color_continuous_scale="Blues")
fig.update_layout(xaxis_tickangle=-45, coloraxis_showscale=False)
fig.show()

In [ ]:
# Revenue by customer state
rev_state = (orders
    .merge(customers[["customer_id","customer_state"]], on="customer_id")
    .merge(items[["order_id","price"]], on="order_id")
    .groupby("customer_state")["price"]
    .sum()
    .sort_values(ascending=False)
    .reset_index())

fig = px.bar(rev_state, x="customer_state", y="price",
             title="Total Revenue by Customer State",
             labels={"price": "Revenue (R$)", "customer_state": "State"},
             color="price", color_continuous_scale="Teal")
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Payment type distribution
pay_type = payments["payment_type"].value_counts().reset_index()
pay_type.columns = ["payment_type", "count"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pay_type.set_index("payment_type")["count"].plot(kind="bar", ax=axes[0],
    color="coral", edgecolor="white")
axes[0].set_title("Orders by Payment Type")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=30)

axes[1].pie(pay_type["count"], labels=pay_type["payment_type"],
            autopct="%1.1f%%", startangle=90)
axes[1].set_title("Payment Type Share")
plt.tight_layout()
plt.show()

## 5. Review Analysis

In [ ]:
# Review score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

score_counts = reviews["review_score"].value_counts().sort_index()
colors = ["#e74c3c","#e67e22","#f1c40f","#2ecc71","#27ae60"]
score_counts.plot(kind="bar", ax=axes[0], color=colors, edgecolor="white")
axes[0].set_title("Review Score Distribution")
axes[0].set_xlabel("Score (1–5)")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

axes[1].pie(score_counts.values, labels=score_counts.index,
            autopct="%1.1f%%", colors=colors, startangle=90)
axes[1].set_title("Score Share")
plt.tight_layout()
plt.show()

print(f"5-star: {(reviews['review_score']==5).sum():,}  ({(reviews['review_score']==5).mean()*100:.1f}%)")
print(f"1-star: {(reviews['review_score']==1).sum():,}  ({(reviews['review_score']==1).mean()*100:.1f}%)")
print(f"Mean:   {reviews['review_score'].mean():.2f}")

In [ ]:
# Review text coverage & length distribution
reviews["has_text"]    = reviews["review_comment_message"].notna()
reviews["text_length"] = reviews["review_comment_message"].str.len().fillna(0)

print(f"Reviews with text:  {reviews['has_text'].sum():,} / {len(reviews):,}  ({reviews['has_text'].mean()*100:.1f}%)")
print(f"Avg text length:    {reviews[reviews['has_text']]['text_length'].mean():.0f} chars")
print(f"Max text length:    {reviews['text_length'].max():.0f} chars")

reviews[reviews["has_text"]]["text_length"].plot(
    kind="hist", bins=60, color="mediumpurple", edgecolor="white")
plt.title("Review Text Length Distribution (chars)")
plt.xlabel("Characters")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 6. Delivery Performance Analysis

In [ ]:
# Late vs on-time delivery analysis
for col in ["order_delivered_customer_date","order_estimated_delivery_date","order_purchase_timestamp"]:
    orders[col] = pd.to_datetime(orders[col])

delivered = orders[orders["order_status"] == "delivered"].copy()
delivered["delivery_days"] = (delivered["order_delivered_customer_date"] -
                               delivered["order_purchase_timestamp"]).dt.days
delivered["is_late"]       = (delivered["order_delivered_customer_date"] >
                               delivered["order_estimated_delivery_date"])

print(f"Delivered orders:   {len(delivered):,}")
print(f"Late deliveries:    {delivered['is_late'].sum():,}  ({delivered['is_late'].mean()*100:.1f}%)")
print(f"On-time deliveries: {(~delivered['is_late']).sum():,}  ({(~delivered['is_late']).mean()*100:.1f}%)")
print(f"Avg delivery days:  {delivered['delivery_days'].mean():.1f}")
print(f"Min delivery days:  {delivered['delivery_days'].min()}")
print(f"Max delivery days:  {delivered['delivery_days'].max()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie([delivered["is_late"].value_counts()[False],
             delivered["is_late"].value_counts()[True]],
            labels=["On Time","Late"], autopct="%1.1f%%",
            colors=["#2ecc71","#e74c3c"], startangle=90)
axes[0].set_title("On-Time vs Late Deliveries")

delivered["delivery_days"].clip(upper=60).plot(kind="hist", bins=40,
    ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("Delivery Days Distribution")
axes[1].set_xlabel("Days from purchase to delivery")
plt.tight_layout()
plt.show()

## 7. Customer & Seller Analysis

In [ ]:
# Customer geographic distribution
cust_state = customers["customer_state"].value_counts().head(15).reset_index()
cust_state.columns = ["state","count"]

fig = px.bar(cust_state, x="state", y="count",
             title="Top 15 States by Customer Count",
             labels={"count":"Customers","state":"State"},
             color="count", color_continuous_scale="Blues")
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Top 10 sellers by revenue
seller_rev = (items
    .groupby("seller_id")["price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index())
seller_rev["seller_id"] = seller_rev["seller_id"].str[:8] + "..."

fig = px.bar(seller_rev, x="seller_id", y="price",
             title="Top 10 Sellers by Revenue",
             labels={"price":"Revenue (R$)","seller_id":"Seller ID (truncated)"},
             color="price", color_continuous_scale="Oranges")
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 8. Pipeline Evaluation

Testing all three pipelines (SQL / RAG / HYBRID) with real queries to demonstrate system correctness.

In [ ]:
# SQL Pipeline Evaluation
from sql.nl_to_sql import nl_to_sql, run_sql

sql_tests = [
    "What are the top 5 product categories by total revenue?",
    "How many orders were delivered late?",
    "What is the average review score by order status?",
    "Which are the top 5 states by number of orders?",
]

sql_results = []
for q in sql_tests:
    sql  = nl_to_sql(q)
    df   = run_sql(sql)
    if "error" in df.columns:
        sql  = nl_to_sql(q, retry_error=df["error"][0])
        df   = run_sql(sql)
    status = "✓ PASS" if "error" not in df.columns else "✗ FAIL"
    sql_results.append({"Question": q, "Status": status, "Rows": len(df)})
    print(f"\n{status}  {q}")
    print(df.to_string(index=False))
    print("─" * 65)

print(f"\nSQL Pipeline Accuracy: {sum(1 for r in sql_results if '✓' in r['Status'])}/{len(sql_tests)}")

In [ ]:
# RAG Pipeline Evaluation
from rag.retriever import FaissRetriever
from llm.sentiment import analyse_sentiment

INDEX_PATH  = "D:/workspace/LLM-PoweredAnalytics/database/faiss.index"
CHUNKS_PATH = "D:/workspace/LLM-PoweredAnalytics/database/chunks.pkl"
retriever   = FaissRetriever(INDEX_PATH, CHUNKS_PATH)

rag_tests = [
    "late delivery damaged product",
    "wrong item received poor quality",
    "excellent product fast shipping great service",
    "missing parts broken on arrival",
]

for q in rag_tests:
    chunks = retriever.retrieve(q, top_k=5)
    result = analyse_sentiment(chunks)
    print(f"\nQuery:     {q}")
    print(f"Sentiment: {result['sentiment'].upper()}")
    print(f"Themes:")
    for t in result["themes"]:
        print(f"  • {t}")
    print("─" * 65)

In [ ]:
# Query Router Accuracy Evaluation
from llm.router import route_query

test_cases = [
    ("What are the top 5 sellers by revenue?",             "SQL"),
    ("What do customers complain about most?",             "RAG"),
    ("How many orders were delivered late?",               "SQL"),
    ("What is the sentiment around electronics products?", "RAG"),
    ("What are reviews saying about top products?",        "HYBRID"),
    ("Which product categories earn the most?",            "SQL"),
    ("What feedback do customers give about packaging?",   "RAG"),
]

correct = 0
print(f"{'Result':6}  {'Expected':8}  {'Got':8}  Question")
print("─" * 70)
for question, expected in test_cases:
    predicted = route_query(question)
    match     = "✓ PASS" if predicted == expected else "✗ FAIL"
    correct  += (predicted == expected)
    print(f"{match}  {expected:8}  {predicted:8}  {question}")

print(f"\nRouter Accuracy: {correct}/{len(test_cases)}  ({correct/len(test_cases)*100:.0f}%)")

In [ ]:
# HYBRID Pipeline Evaluation
from llm.synthesizer import synthesize

hybrid_q = "What are the top selling product categories and what do customers say about them?"

print(f"Question: {hybrid_q}\n")

# SQL part
sql  = nl_to_sql(hybrid_q)
df   = run_sql(sql)
if "error" in df.columns:
    sql = nl_to_sql(hybrid_q, retry_error=df["error"][0])
    df  = run_sql(sql)

print("── SQL Result:")
print(df.to_string(index=False))

# RAG part
chunks     = retriever.retrieve(hybrid_q, top_k=5)
rag_result = analyse_sentiment(chunks)
print(f"\n── RAG Sentiment: {rag_result['sentiment'].upper()}")
print(f"── Themes: {rag_result['themes']}")

# Synthesize
answer = synthesize(hybrid_q, df.to_string(index=False), rag_result)
print(f"\n── Combined Insight:\n{answer}")